# 08 · Business Interpretation, Recommendations and Reproducibility
### IT3091 Machine Learning · Group 2026-AI-42 · Evaluation, recommendation & documentation lead

Consolidation of evaluation tables from notebooks 02–07, summarizing strategic recommendations, limitations, project decisions, and environment configuration.

In [1]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import json, platform
import numpy as np, pandas as pd
from IPython.display import display
import sklearn

from src.config import *
from src.data.loading import fingerprint
from src.data.prepare import load_modelling_data

pd.set_option("display.max_columns", 60, "display.width", 170, "display.max_colwidth", 140)
data = load_modelling_data()
y_test, y_train = data["y_test"], data["y_train"]
T = lambda name, **kw: pd.read_csv(TABLES_DIR / name, **kw)
params = json.loads((ARTIFACTS_DIR / "tuned_params.json").read_text())
labels = json.loads((ARTIFACTS_DIR / "policy_labels.json").read_text())
TUNED_NAME = params["name"]

Loading local file: bank-additional-full.csv
Rows: raw 41,188 -> cleaned 41,176 | train 32,940 (rate 6.38%) | pilot 500 | test 7,736 (rate 32.41%)


---
## 1 · Key numbers for the report
*Test = the later time period, used once for the final evaluation. Cost and value figures are illustrative.*

In [2]:
holdout = T("holdout_results.csv", index_col=0)
policies = T("policy_comparison.csv", index_col=0)
blocks = T("test_blocks.csv", index_col=0)
leak = T("preprocessing_leakage_duration.csv", index_col=0)
split_opt = T("split_optimism.csv", index_col=0)

best_row = holdout.loc[TUNED_NAME]
top, rnd, allc = policies.loc[labels["P_BUDGET"]], policies.loc[labels["P_RANDOM"]], policies.loc[labels["P_ALL"]]
print("=== Key numbers for the report ===")
print(f"FINAL MODEL: {TUNED_NAME} (selected on forward-chaining PR-AUC in the training period, notebooks 03-04)")
print(f"CLIENT-LEVEL SKILL (within time blocks): ROC-AUC {blocks['auc'].mean():.3f} | lift@{CALL_BUDGET:.0%} {blocks['lift_at_budget'].mean():.2f}x")
print(f"Capacity policy (top {CALL_BUDGET:.0%} of each block): captures {top.subscribers_captured_pct:.1f}% of subscribers "
      f"(precision {top.precision:.3f} vs {y_test.mean():.3f} at random = {top.precision / y_test.mean():.2f}x)")
print(f"Profit per client at a {CALL_BUDGET:.0%} call capacity: model {top.profit_per_client:.3f} vs random {rnd.profit_per_client:.3f} | call everyone {allc.profit_per_client:.3f}")
print(f"GLOBAL ranking over the whole test period: ROC-AUC {best_row.roc_auc:.3f} | PR-AUC {best_row.pr_auc:.3f} | lift@{CALL_BUDGET:.0%} {best_row.lift_at_budget:.2f}x")
print(f"Leakage check: ROC-AUC with duration {leak.iloc[0].roc_auc:.3f} vs without {leak.iloc[1].roc_auc:.3f}")
opt = split_opt.iloc[:3][["roc_auc", "lift_at_budget"]].mean() - split_opt.iloc[3][["roc_auc", "lift_at_budget"]]
print(f"Optimism of a stratified random split vs the chronological hold-out: ROC-AUC {opt['roc_auc']:+.3f}, lift {opt['lift_at_budget']:+.2f}x")

=== Key numbers for the report ===
FINAL MODEL: Tuned Logistic regression (selected on forward-chaining PR-AUC in the training period, notebooks 03-04)
CLIENT-LEVEL SKILL (within time blocks): ROC-AUC 0.609 | lift@20% 1.35x
Capacity policy (top 20% of each block): captures 26.9% of subscribers (precision 0.436 vs 0.324 at random = 1.35x)
Profit per client at a 20% call capacity: model 0.673 vs random 0.448 | call everyone 2.241
GLOBAL ranking over the whole test period: ROC-AUC 0.689 | PR-AUC 0.499 | lift@20% 1.72x
Leakage check: ROC-AUC with duration 0.953 vs without 0.671
Optimism of a stratified random split vs the chronological hold-out: ROC-AUC +0.073, lift +1.09x


---
## 2 · Recommendations

1. **Deploy a ranked-call policy rather than a fixed binary cutoff:** Prioritize outreach in descending score order up to available call capacity.
2. **Exclude post-call duration:** Avoid target leakage from `duration` to ensure genuine pre-call predictive capability.
3. **Utilize segment profiles for campaign scripting:** Combine client-level priority scores with segment-specific scripts and agent assignment.
4. **Track temporal drift:** Regularly re-estimate the base rate using rolling intercepts and schedule periodic model retraining.
5. **Cap repeat contacts:** Limit successive contact attempts to avoid diminishing return on call effort.
6. **Maintain human oversight:** Review ethical audits and ensure fair treatment across demographic groups.

### Limitations and Risks
* Dataset originates from a Portuguese banking institution (2008–2010); findings should be validated in current market conditions.
* Temporal drift in subscription rates requires continual monitoring.
* Macroeconomic indicators are excluded to prevent reliance on transient temporal proxies.
* Lack of unique client identifiers prevents tracking repeat individuals across time periods.
* The model captures correlation rather than causal uplift.

### Business Value
Prioritizing leads using model scores significantly outperforms random outreach at the target budget, improving conversion efficiency and contact center resource allocation.

---
## 3 · Decision log (project level)
Group-level decisions with the evidence and where it lives. The detailed preprocessing log is in `docs/preprocessing_log.md`.

| # | Decision | Options considered | Choice | Evidence |
|---|---|---|---|---|
| D1 | Primary lens | response prediction / segmentation / profile analysis | Campaign response prediction | Labelled outcome, costly binary decision, clear baseline |
| D2 | Secondary lens | none / segmentation | Segment prioritisation, only to operationalise scores | Notebook 07 §1 |
| D3 | `duration` | keep / drop | Drop (post-call) | Notebook 02 §7.1 |
| D4 | `unknown` values | drop / impute / own category | Own category; education → NaN + indicator | Notebook 01 §3, 02 §3 |
| D5 | `pdays = 999` | numeric / NaN / flag | Flag + 0 | Notebook 01 §3.6, 02 §3.3 |
| D6 | Evaluation metrics | accuracy / ROC-AUC / PR-AUC / lift / profit | PR-AUC, ROC-AUC, budget lift / recall / precision, profit | Notebook 01 §4, 03 |
| D7 | Validation design | stratified random / chronological | Stratified CV and forward-chaining for selection; **chronological** hold-out for the final result | Notebook 01 §9, 05 §4 |
| D8 | Imbalance handling | none / weights / SMOTE / SMOTE-Tomek | Simplest option unless SMOTE-Tomek improves ranking | Notebook 04 §1-2 |
| D9 | Final model | LR / RF / HGB / MLP / challengers | Read from notebooks 03-05 (printed in §1 above) | Notebooks 03-05 |
| D10 | Probability calibration | none / isotonic / Platt / rolling intercept | Rolling intercept + rank-based policy | Notebook 06 §2 |
| D11 | Dataset version and macro indicators | `bank-full` / additional ± macro | Rule fixed in advance; outcome in notebook 03 §5 | Notebook 03 §5 |
| D12 | Proposal feature families | baseline / each / all | Adopt only on forward-chaining evidence | Notebook 02 §6 |
| D13 | Extra models | keep incumbent / challenger | Challenger replaces incumbent only with higher mean PR-AUC and ≥ 3 of 4 folds | Notebook 03 §3 |
| D14 | Operating point | static threshold / break-even / rank-based call volume | Rank-based call volume | Notebook 06 §5 |
| D15 | Explanation | permutation only / add TreeSHAP | Report both | Notebook 07 §2 |
| D16 | Sensitive features | keep / drop | Exclude from production model due to negligible accuracy loss | Notebook 07 §3 |


---
## 4 · Reproducibility

In [3]:
print("Python        :", platform.python_version())
print("pandas / numpy:", pd.__version__, "/", np.__version__)
print("scikit-learn  :", sklearn.__version__)
print("Random seed   :", SEED)
print("Dataset       : UCI Bank Marketing (id 222), bank-additional-full.csv")
print("Shape         :", data["raw"].shape, "| after de-duplication:", data["df"].shape)
print("Fingerprint   :", fingerprint(data["raw"]))
print("Business config: budget =", CALL_BUDGET, "| cost =", COST_PER_CALL, "| value =", VALUE_PER_SUBSCRIPTION)
print("Macro excluded:", EXCLUDE_MACRO, "| adopted feature families:", ADOPTED_FAMILIES or "none")

Python        : 3.12.9
pandas / numpy: 3.0.1 / 2.4.3
scikit-learn  : 1.8.0
Random seed   : 42
Dataset       : UCI Bank Marketing (id 222), bank-additional-full.csv
Shape         : (41188, 21) | after de-duplication: (41176, 21)
Fingerprint   : 00de77bb3ece5d5fdfdd09ec12392284dac89d6d877ec1659092eb5b3522abe1
Business config: budget = 0.2 | cost = 1.0 | value = 10.0
Macro excluded: True | adopted feature families: none


### References
* Moro, S., Cortez, P., & Rita, P. (2014). A data-driven approach to predict the success of bank telemarketing. *Decision Support Systems, 62*, 22-31. Dataset: Moro, S., Rita, P., & Cortez, P. (2014). *Bank Marketing* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306
* Batista, G. E. A. P. A., Prati, R. C., & Monard, M. C. (2004). A study of the behavior of several methods for balancing machine learning training data. *ACM SIGKDD Explorations Newsletter, 6*(1), 20-29.
* Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794.
* Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems 30*.
* Lundberg, S. M., et al. (2020). From local explanations to global understanding with explainable AI for trees. *Nature Machine Intelligence, 2*, 56-67.
